In [110]:
import torch
import transformers

Getting the model...

In [163]:
model_name = "distilbert/distilgpt2"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name, device_map='auto')
model = transformers.AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

paragraph = r"""Teddy swiveled his chair and looked out the window to the sky beyond. Night was edging in. "What must it be like?” He pondered. "He's stuck out there. He thinks he's totally alone and that we all gave up on him. What kind of effect does that have on a man's psychology?" He turned back to Venkat. "I wonder what he's thinking right now." LOG ENTRY: SOL 61 How come Aquaman can control whales? They're mammals! Makes no sense"""
# paragraph = r"""The quick brown fox jumped over the lazy dog"""

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [179]:
def ppl(tokens):
    N = len(tokens[0])

    with torch.inference_mode():
        model.eval()
        out = model(**tokens)
        p_dist = torch.nn.functional.softmax(out.logits, dim=-1)

    H = 0
    for i, token in zip(range(N), tokens["input_ids"][0]):
        p = p_dist[0, i, token]
        H += torch.log2(p)
    H = -H / N

    ppl = 2 ** H
    return ppl

tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
ppl(tokens)

tensor(47490.1992)

In [182]:
shuffled_tokens = tokenizer(paragraph, return_tensors="pt").to(model.device)
n = shuffled_tokens['input_ids'].shape[-1]
shuffled_tokens['input_ids'] = shuffled_tokens['input_ids'][...,torch.randperm(n)]
ppl(shuffled_tokens)

tensor(2759.1021)